In [1]:
import pandas as pd
import torch
import numpy as np
import os
from torch.utils.data import DataLoader, Dataset
import re
import random
import gc
import json
from collections import Counter, defaultdict
from transformers import (
    BertConfig,
    BertForMaskedLM,
    Trainer,
    TrainingArguments,
    set_seed
)

import torch.nn as nn

from tqdm.auto import tqdm

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
!unzip /content/bert_preprocess.zip -d ../


Archive:  /content/bert_preprocess.zip
   creating: ../content/financial_bert_final/
  inflating: ../content/financial_bert_final/vocab.json  
  inflating: ../content/financial_bert_final/preprocessing.json  


In [4]:
df = pd.read_csv("/content/transactions.tgz",compression='gzip', nrows=8_000_000)
df

,card_transaction.v1.csv,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7999995,680,2,2003,10,1,07:30,$61.82,Swipe Transaction,-7566024815690246185,Miami,FL,33179.0,8021,NaN,No
7999996,680,2,2003,10,2,06:58,$2.81,Swipe Transaction,-2744911404133435018,Miami,FL,33183.0,5812,NaN,No
7999997,680,2,2003,10,2,13:14,$7.20,Swipe Transaction,4722913068560264812,Miami,FL,33179.0,5411,NaN,No
7999998,680,2,2003,10,2,13:39,$53.76,Swipe Transaction,1799189980464955940,Miami,FL,33179.0,5499,NaN,No


In [5]:
COLUMN_MAP = {
    "card_transaction.v1.csv": "user",
    "Card": "card",
    "Year": "year",
    "Month": "month",
    "Day": "day",
    "Time": "time",
    "Amount": "amount",
    "Use Chip": "use_chip",
    "Merchant Name": "merchant_name",
    "Merchant City": "merchant_city",
    "Merchant State": "merchant_state",
    "Zip": "zip",
    "MCC": "mcc",
    "Errors?": "errors",
    "Is Fraud?": "is_fraud",
}

df = df.rename(columns=COLUMN_MAP)
df

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7999995,680,2,2003,10,1,07:30,$61.82,Swipe Transaction,-7566024815690246185,Miami,FL,33179.0,8021,NaN,No
7999996,680,2,2003,10,2,06:58,$2.81,Swipe Transaction,-2744911404133435018,Miami,FL,33183.0,5812,NaN,No
7999997,680,2,2003,10,2,13:14,$7.20,Swipe Transaction,4722913068560264812,Miami,FL,33179.0,5411,NaN,No
7999998,680,2,2003,10,2,13:39,$53.76,Swipe Transaction,1799189980464955940,Miami,FL,33179.0,5499,NaN,No


In [6]:
date = pd.to_datetime(
    dict(
        year=df["year"],
        month=df["month"],
        day=df["day"]
    )
)
time = pd.to_timedelta(
    df["time"].astype(str) + ":00"
)
df["timestamp"] = date + time
df

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud,timestamp
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No,2002-09-01 06:21:00
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No,2002-09-01 06:42:00
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No,2002-09-02 06:22:00
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No,2002-09-02 17:45:00
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No,2002-09-03 06:23:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7999995,680,2,2003,10,1,07:30,$61.82,Swipe Transaction,-7566024815690246185,Miami,FL,33179.0,8021,NaN,No,2003-10-01 07:30:00
7999996,680,2,2003,10,2,06:58,$2.81,Swipe Transaction,-2744911404133435018,Miami,FL,33183.0,5812,NaN,No,2003-10-02 06:58:00
7999997,680,2,2003,10,2,13:14,$7.20,Swipe Transaction,4722913068560264812,Miami,FL,33179.0,5411,NaN,No,2003-10-02 13:14:00
7999998,680,2,2003,10,2,13:39,$53.76,Swipe Transaction,1799189980464955940,Miami,FL,33179.0,5499,NaN,No,2003-10-02 13:39:00


In [7]:
df["amount_numeric"] = df["amount"].astype(str).str.replace("$","").str.replace(",","")
df["amount_numeric"] = pd.to_numeric(df["amount_numeric"])
df.head()

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud,timestamp,amount_numeric
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No,2002-09-01 06:21:00,134.09
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No,2002-09-01 06:42:00,38.48
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No,2002-09-02 06:22:00,120.34
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No,2002-09-02 17:45:00,128.95
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No,2002-09-03 06:23:00,104.71


In [8]:
df = df.sort_values(["user", "timestamp"]).reset_index(drop=True)

In [9]:
df["hour"] = df["timestamp"].dt.hour

df["day_of_week"] = (
    df["timestamp"].dt.dayofweek
)

df["day_of_month"] = (
    df["timestamp"].dt.day
)

df["calendar_month"] = (
    df["timestamp"].dt.month
)

df["previous_time"] = (
    df.groupby("user")["timestamp"]
    .diff()
    .dt
    .total_seconds()
    .div(60)
)
df["previous_time"] = (
    df["previous_time"]
    .fillna(0)
    .clip(lower=0,upper=60*24*30)
)

df

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,...,mcc,errors,is_fraud,timestamp,amount_numeric,hour,day_of_week,day_of_month,calendar_month,previous_time
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,...,5300,NaN,No,2002-09-01 06:21:00,134.09,6,6,1,9,0.0
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,...,5411,NaN,No,2002-09-01 06:42:00,38.48,6,6,1,9,21.0
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,...,5411,NaN,No,2002-09-02 06:22:00,120.34,6,0,2,9,1420.0
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,...,5651,NaN,No,2002-09-02 17:45:00,128.95,17,0,2,9,683.0
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,...,5912,NaN,No,2002-09-03 06:23:00,104.71,6,1,3,9,758.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7999995,680,1,2020,2,26,16:10,$26.98,Chip Transaction,272399770636553347,Miami,...,5411,NaN,No,2020-02-26 16:10:00,26.98,16,2,26,2,559.0
7999996,680,1,2020,2,27,12:41,$8.13,Chip Transaction,272399770636553347,Miami,...,5411,NaN,No,2020-02-27 12:41:00,8.13,12,3,27,2,1231.0
7999997,680,0,2020,2,28,06:49,$2.72,Chip Transaction,1799189980464955940,Miami,...,5499,NaN,No,2020-02-28 06:49:00,2.72,6,4,28,2,1088.0
7999998,680,1,2020,2,28,12:43,$6.94,Chip Transaction,272399770636553347,Miami,...,5411,NaN,No,2020-02-28 12:43:00,6.94,12,4,28,2,354.0


In [10]:
# df["fraud_label"] = (
#     df["is_fraud"].astype(str)
#     .str.strip()
#     .str.upper()
#     .map({
#         "YES": 1,
#         "NO": 0
#     })

# )

In [11]:
train_df = df[
    df["timestamp"] < pd.Timestamp("2017-01-01")
].copy()

val_df = df[
    (df["timestamp"] >= pd.Timestamp("2017-01-01"))
    & (df["timestamp"] < pd.Timestamp("2019-01-01"))
].copy()

test_df = df[
    df["timestamp"] >= pd.Timestamp("2019-01-01")
].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 6198642
Validation: 1123895
Test: 677463


In [12]:
# ENCODED_PATH = (
#     "/content/encoded_transactions.uint16.mmap"
# )


# encoded_transactions = np.memmap(

#     ENCODED_PATH,

#     dtype=np.uint16,

#     mode="w+",

#     shape=(
#         len(df),
#         TOKENS_PER_TRANSACTION
#     )

# )

In [13]:
# encoded_transactions.flush()

# print(
#     "Encoded transaction shape:",
#     encoded_transactions.shape
# )

In [14]:
import gc
del df
gc.collect()

212

In [15]:
dataset = [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]

for name, part in dataset:
    print(
        name, part["is_fraud"].value_counts(
            normalize=True
        )
    )

train is_fraud
No     0.998717
Yes    0.001283
Name: proportion, dtype: float64
validation is_fraud
No     0.999197
Yes    0.000803
Name: proportion, dtype: float64
test is_fraud
No     0.999074
Yes    0.000926
Name: proportion, dtype: float64


In [16]:
FOUNDATION_DIR = "/content/financial_bert_final"

VOCAB_PATH = os.path.join(FOUNDATION_DIR,"vocab.json")

PREPROCESSING_PATH = os.path.join(FOUNDATION_DIR,"preprocessing.json")


In [17]:
with open(VOCAB_PATH, "r") as f:
    token_to_id = json.load(f)


with open(PREPROCESSING_PATH, "r") as f:
    preprocessing = json.load(f)


print("Vocabulary size:", len(token_to_id))

print(
    "Preprocessing keys:",
    preprocessing.keys()
)

Vocabulary size: 22223
Preprocessing keys: dict_keys(['amount_boundaries', 'delta_boundaries', 'timestamp_boundaries', 'transactions_per_sequence', 'tokens_per_transaction', 'max_length'])


In [18]:
amount_boundaries = np.asarray(
    preprocessing["amount_boundaries"],
    dtype=np.float64
)

delta_boundaries = np.asarray(
    preprocessing["delta_boundaries"],
    dtype=np.float64
)

if "timestamp_boundaries" in preprocessing:

    timestamp_boundaries = np.asarray(
        preprocessing["timestamp_boundaries"],
        dtype=np.float64
    )

else:

    print("timestamp_boundaries missing from preprocessing.json.")

In [19]:
def fit_quantile_boundaries(values, n_bins):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    quantiles = np.linspace(0, 1, n_bins + 1)[1:-1]
    boundaries = np.quantile(values, quantiles)
    boundaries = np.unique(boundaries)
    return boundaries


def apply_quantization(values, boundaries):
    values = np.asarray(values, dtype=np.float64)
    ind = np.digitize(values, boundaries, right=False)
    return ind.astype(dtype=np.int32)


In [20]:
for name, part in dataset:
    part["amount_bin"] = apply_quantization(
        part["amount_numeric"],
        amount_boundaries
    )


    part["delta_bin"] = apply_quantization(
        part["previous_time"],
        delta_boundaries
    )


    timestamp_seconds = part["timestamp"].astype(np.int64) // 10**9

    part["timestamp_bin"] = apply_quantization(
        timestamp_seconds,
        timestamp_boundaries
    )

## Creating the tokenizer class

In [21]:
class TransactionTokenizer:

    """
    Tokenizer for structured financial transactions.

    Converts:

        transaction row
            ↓
        field=value tokens
            ↓
        vocabulary IDs

    Uses the SAME vocabulary as the pretrained
    foundation model.
    """


    def __init__(
        self,
        token_to_id,
        include_timestamp=True
    ):

        self.token_to_id = token_to_id

        self.id_to_token = {
            idx: token
            for token, idx
            in token_to_id.items()
        }

        self.include_timestamp = include_timestamp


    @staticmethod
    def clean(value):

        if pd.isna(value):
            return "NONE"

        value = str(value).strip().upper()

        value = re.sub(r"\s+", "_",value)

        value = value.replace("=","_")

        return value


    @staticmethod
    def clean_zip(value):

        if pd.isna(value):
            return "NONE"

        try:

            number = float(value)

            if number.is_integer():

                return str(int(number))

        except (TypeError, ValueError):

            pass


        return TransactionTokenizer.clean(value)


    @staticmethod
    def get_field(token):

        if token.startswith("["):
            return None

        if "=" not in token:
            return None

        return token.split("=",1)[0]


    def tokenize_transaction(self,row):

        tokens = [
            "[TXN]",
            f"CARD={self.clean(row.card)}"
        ]

        if self.include_timestamp:
            tokens.append(f"TIMESTAMP={int(row.timestamp_bin)}")


        tokens.extend([
            f"HOUR={int(row.hour)}",
            f"DOW={int(row.day_of_week)}",
            f"MONTH={int(row.calendar_month)}",
            f"DOM={int(row.day_of_month)}",
            f"DELTA={int(row.delta_bin)}",
            f"AMOUNT={int(row.amount_bin)}",
            f"CHANNEL={self.clean(row.use_chip)}",
            f"MERCHANT={self.clean(row.merchant_name)}",
            f"CITY={self.clean(row.merchant_city)}",
            f"STATE={self.clean(row.merchant_state)}",
            f"ZIP={self.clean_zip(row.zip)}",
            f"MCC={self.clean(row.mcc)}",
            f"ERROR={self.clean(row.errors)}",
        ])


        return tokens

    def encode_token(
        self,
        token
    ):

        # Known token

        if token in self.token_to_id:

            return self.token_to_id[
                token
            ]


        # Try field-specific unknown

        field = self.get_field(token)

        if field is not None:

            unknown_token = (
                f"[UNK_{field}]"
            )


            if unknown_token in self.token_to_id:

                return self.token_to_id[unknown_token]


        # Global unknown

        return self.token_to_id["[UNK]"]



    def encode_transaction(self, row):

        tokens = self.tokenize_transaction(row)

        return [
            self.encode_token(token)
            for token in tokens
        ]


    def decode(self, ids):
        return [
            self.id_to_token.get(
                int(idx),
                "[UNK]"
            )
            for idx in ids

        ]


    @property
    def cls_token_id(self):
        return self.token_to_id["[CLS]"]


    @property
    def sep_token_id(self):
        return self.token_to_id["[SEP]"]

    @property
    def pad_token_id(self):
        return self.token_to_id["[PAD]"]

    @property
    def unk_token_id(self):
        return self.token_to_id["[UNK]"]

    @property
    def txn_token_id(self):
        return self.token_to_id["[TXN]"]


    def __len__(self):
        return len(self.token_to_id)

In [22]:
tokenizer = TransactionTokenizer(
    token_to_id=token_to_id,
    include_timestamp=True
)

In [23]:
# for index, row in enumerate(
#     tqdm(
#         df.itertuples(
#             index=False
#         ),

#         total=len(df),
#         desc="Encoding transactions"
#     )
# ):
#     encoded_transactions[
#         index
#     ] = np.asarray(
#         tokenizer.encode_transaction(row),
#         dtype=np.uint16
#     )

In [24]:
# encoded_transactions.flush()

# print(
#     "Encoded transaction shape:",
#     encoded_transactions.shape
# )

In [25]:
# encoded_transactions[
#         index
#     ] = np.asarray(

#         tokenizer.encode_transaction(
#             row
#         ),

#         dtype=np.uint16

#     )

In [26]:
KEEP_COLUMNS = [
    "user",
    "card",

    "timestamp",

    "hour",
    "day_of_week",
    "calendar_month",
    "day_of_month",

    "delta_bin",
    "amount_bin",
    "timestamp_bin",

    "use_chip",
    "merchant_name",
    "merchant_city",
    "merchant_state",
    "zip",
    "mcc",
    "errors",
    "is_fraud"

]

In [27]:
all_df = pd.concat(
    [
        train_df[KEEP_COLUMNS],
        val_df[KEEP_COLUMNS],
        test_df[KEEP_COLUMNS]
    ],
    ignore_index=True,
    copy=False
)

In [28]:
all_df = (
    all_df
    .sort_values(
        [
            "user",
            "timestamp"
        ]
    )
    .reset_index(
        drop=True
    )
)

In [29]:
sample_row = next(train_df.itertuples(index=False))

sample_tokens = tokenizer.tokenize_transaction(sample_row)

sample_ids = tokenizer.encode_transaction(sample_row)


print(sample_tokens)

print(sample_ids)

print(tokenizer.decode(sample_ids))

['[TXN]', 'CARD=0', 'TIMESTAMP=2', 'HOUR=6', 'DOW=6', 'MONTH=9', 'DOM=1', 'DELTA=0', 'AMOUNT=29', 'CHANNEL=SWIPE_TRANSACTION', 'MERCHANT=3527213246127876953', 'CITY=LA_VERNE', 'STATE=CA', 'ZIP=91750', 'MCC=5300', 'ERROR=NONE']
[5, 53, 12967, 5152, 5122, 12828, 5093, 5088, 37, 61, 5544, 106, 12835, 13064, 5174, 5127]
['[TXN]', 'CARD=0', 'TIMESTAMP=2', 'HOUR=6', 'DOW=6', 'MONTH=9', 'DOM=1', 'DELTA=0', 'AMOUNT=29', 'CHANNEL=SWIPE_TRANSACTION', 'MERCHANT=3527213246127876953', 'CITY=LA_VERNE', 'STATE=CA', 'ZIP=91750', 'MCC=5300', 'ERROR=NONE']


In [30]:
del train_df
del val_df
del test_df

gc.collect()

0

In [31]:
all_df["fraud_label"] = (
    all_df["is_fraud"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({
        "YES": 1,
        "NO": 0

    })

)

In [32]:
TOKENS_PER_TRANSACTION = len(sample_ids)

print(
    "Tokens per transaction:",
    TOKENS_PER_TRANSACTION
)

Tokens per transaction: 16


In [33]:
ENCODED_PATH = "/content/encoded_transactions.uint16.mmap"

encoded_transactions = np.memmap(
    ENCODED_PATH,
    dtype=np.uint16,
    mode="w+",
    shape=(len(all_df), TOKENS_PER_TRANSACTION)
)

In [34]:
for index, row in enumerate(

    tqdm(
        all_df.itertuples(
            index=False
        ),
        total=len(all_df),
        desc="Encoding transactions"
    )
):

    encoded_transactions[index] = np.asarray(
        tokenizer.encode_transaction(row),
        dtype=np.uint16
    )

Encoding transactions:   0%|          | 0/8000000 [00:00<?, ?it/s]

In [35]:
encoded_transactions.flush()

print(
    "Encoded transaction shape:",
    encoded_transactions.shape
)

Encoded transaction shape: (8000000, 16)


In [36]:
TRANSACTIONS_PER_SEQUENCE = 16

HISTORY_REQUIRED = TRANSACTIONS_PER_SEQUENCE - 1

In [37]:
users = all_df["user"].to_numpy()

In [38]:
user_starts = np.flatnonzero(
    np.r_[True, users[1:] != users[:-1]]
)

In [39]:
user_ends = np.r_[user_starts[1:], len(all_df)]

In [40]:
target_parts = []

for start, end in zip(
    user_starts,
    user_ends
):

    first_target = start + HISTORY_REQUIRED

    if first_target >= end:
        continue

    targets = np.arange(first_target,end, dtype=np.int32)

    target_parts.append(targets)

In [41]:
all_targets = np.concatenate(target_parts)

del target_parts
gc.collect()

print("Eligible fraud examples:", len(all_targets))

Eligible fraud examples: 7989785


In [42]:
timestamps = all_df["timestamp"].to_numpy()

labels = all_df["fraud_label"].to_numpy(dtype=np.uint8)

In [43]:
TRAIN_END = np.datetime64("2017-01-01")

VALIDATION_END = np.datetime64("2019-01-01")

In [44]:
target_timestamps = timestamps[all_targets]

In [45]:
train_mask = target_timestamps < TRAIN_END

validation_mask = (target_timestamps>= TRAIN_END) & (target_timestamps < VALIDATION_END)

test_mask = target_timestamps >= VALIDATION_END

In [46]:
train_targets_all = all_targets[train_mask]
validation_targets = all_targets[validation_mask]
test_targets = all_targets[test_mask]

In [47]:
print("Training targets:", len(train_targets_all))

print("Validation targets:", len(validation_targets))

print("Test targets:", len(test_targets))

Training targets: 6190946
Validation targets: 1123746
Test targets: 675093


In [48]:
SEED = 42

NEGATIVE_TO_POSITIVE_RATIO = 10

rng = np.random.default_rng(SEED)

In [49]:
train_y_all = labels[train_targets_all]

In [50]:
positive_targets = train_targets_all[train_y_all == 1]

negative_targets = train_targets_all[train_y_all == 0]

In [51]:
print("Fraud:", len(positive_targets))

print("Normal:", len(negative_targets))

Fraud: 7953
Normal: 6182993


In [52]:
number_negatives = min(
    len(negative_targets),
    len(positive_targets) * NEGATIVE_TO_POSITIVE_RATIO
)

In [53]:
sampled_negative_targets = rng.choice(
    negative_targets,
    size=number_negatives,
    replace=False
)

In [54]:
train_targets = np.concatenate([positive_targets,sampled_negative_targets])

In [55]:
rng.shuffle(
    train_targets
)

In [56]:
def show_distribution(name, targets):

    y = labels[targets]
    print(f"\n{name}")
    print("Transactions:",len(y))
    print("Fraud:",int(y.sum()))
    print("Fraud rate:", f"{y.mean():.6%}")


show_distribution("TRAIN",train_targets)
show_distribution("VALIDATION",validation_targets)
show_distribution("TEST",test_targets)


TRAIN
Transactions: 87483
Fraud: 7953
Fraud rate: 9.090909%

VALIDATION
Transactions: 1123746
Fraud: 903
Fraud rate: 0.080356%

TEST
Transactions: 675093
Fraud: 627
Fraud rate: 0.092876%


In [57]:
class FraudWindowDataset(Dataset):

    def __init__(
        self,
        targets,
        labels,
        encoded_path,
        number_rows,
        tokens_per_transaction,
        transactions_per_sequence,
        tokenizer
    ):

        self.targets = np.asarray(targets, dtype=np.int32)

        self.labels = labels

        self.encoded_path = encoded_path

        self.number_rows = number_rows

        self.tokens_per_transaction = tokens_per_transaction

        self.transactions_per_sequence = transactions_per_sequence

        self.tokenizer = tokenizer

        self._mmap = None


    def _open_memmap(self):

        if self._mmap is None:
            self._mmap = np.memmap(
                self.encoded_path,
                dtype=np.uint16,
                mode="r",
                shape=(
                    self.number_rows,
                    self.tokens_per_transaction
                )

            )


    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        self._open_memmap()

        target_index = int(self.targets[index])
        start_index = (
            target_index
            - self.transactions_per_sequence
            + 1
        )

        transaction_ids = (
            self._mmap[
                start_index:
                target_index + 1
            ]
            .reshape(-1)
            .astype(
                np.int64
            )
        )

        # [CLS] history/current sequence [SEP]

        input_ids = np.concatenate([
            np.asarray(
                [self.tokenizer.cls_token_id],
                dtype=np.int64
            ),

            transaction_ids,

            np.asarray(
                [self.tokenizer.sep_token_id],
                dtype=np.int64
            )

        ])


        attention_mask = np.ones(
            len(input_ids),
            dtype=np.int64
        )

        label = float(self.labels[target_index])

        return {
            "input_ids":torch.from_numpy(input_ids),
            "attention_mask": torch.from_numpy(attention_mask),

            "labels":torch.tensor(label, dtype=torch.float32)

        }

In [58]:
train_dataset = FraudWindowDataset(
    targets=train_targets,
    labels=labels,
    encoded_path=ENCODED_PATH,
    number_rows=len(all_df),
    tokens_per_transaction=TOKENS_PER_TRANSACTION,
    transactions_per_sequence=TRANSACTIONS_PER_SEQUENCE,
    tokenizer=tokenizer
)

In [59]:
validation_dataset = FraudWindowDataset(
    targets=validation_targets,
    labels=labels,
    encoded_path=ENCODED_PATH,
    number_rows=len(all_df),
    tokens_per_transaction=TOKENS_PER_TRANSACTION,
    transactions_per_sequence=TRANSACTIONS_PER_SEQUENCE,
    tokenizer=tokenizer
)

In [60]:
test_dataset = FraudWindowDataset(
    targets=test_targets,
    labels=labels,
    encoded_path=ENCODED_PATH,
    number_rows=len(all_df),
    tokens_per_transaction=TOKENS_PER_TRANSACTION,
    transactions_per_sequence=TRANSACTIONS_PER_SEQUENCE,
    tokenizer=tokenizer
)

In [61]:
BATCH_SIZE = 32
pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory
)

In [62]:
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory
)

In [63]:
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory
)

In [64]:
class FinancialBERTFraudDetector(nn.Module):

    def __init__(
        self,
        encoder,
        hidden_size,
        head_type="lstm",
        lstm_hidden_size=128,
        dropout=0.2
    ):

        super().__init__()

        self.encoder = encoder

        self.hidden_size = hidden_size

        self.head_type = head_type

        self.encoder_frozen = True

        self.freeze_encoder()

        # LSTM head

        if head_type == "lstm":
            self.lstm = nn.LSTM(
                input_size=hidden_size,
                hidden_size=lstm_hidden_size,
                num_layers=1,
                batch_first=True,
                bidirectional=False

            )

            classifier_input = lstm_hidden_size

        # MLP baselines

        elif head_type in {
            "current_mlp",
            "mean_mlp",
            "cls_mlp"
        }:

            classifier_input = hidden_size


        else:
            raise ValueError(f"Unknown head: {head_type}")

        self.classifier = nn.Sequential(
            nn.Linear(classifier_input,64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64,1)
        )


    # Freeze encoder

    def freeze_encoder(self):

        for parameter in self.encoder.parameters():
            parameter.requires_grad = False

        self.encoder_frozen = True


    # Fine-tune final BERT layer

    def unfreeze_last_n_layers(self,n=1):

        for parameter in self.encoder.parameters():

            parameter.requires_grad = False

        for layer in self.encoder.encoder.layer[-n:]:

            for parameter in layer.parameters():
                parameter.requires_grad = True

        self.encoder_frozen = False


    def encode(
        self,
        input_ids,
        attention_mask
    ):
        if self.encoder_frozen:

            self.encoder.eval()

            with torch.no_grad():
                outputs = self.encoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )


        else:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )


        return outputs.last_hidden_state

    def transaction_pool(
        self,
        hidden
    ):

        # Remove CLS and SEP

        hidden = hidden[:, 1:-1, :]

        batch_size = hidden.shape[0]

        hidden = hidden.reshape(
            batch_size,
            TRANSACTIONS_PER_SEQUENCE,
            TOKENS_PER_TRANSACTION,
            self.hidden_size
        )

        field_hidden = hidden[:, :, 1:, :]

        # Mean pooling within EACH transaction

        transaction_embeddings = field_hidden.mean(dim=2)


        return (
            transaction_embeddings
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        hidden = self.encode(
            input_ids,
            attention_mask
        )

        # CLS baseline

        if self.head_type == "cls_mlp":
            representation = hidden[:, 0, :]

        else:
            transaction_embeddings = (
                self.transaction_pool(hidden)
            )

            # Current transaction only

            if self.head_type == "current_mlp":

                representation = transaction_embeddings[:, -1, :]

            # Mean across whole history

            elif self.head_type == "mean_mlp":
                representation = transaction_embeddings.mean(
                    dim=1
                )

            # Temporal LSTM

            elif self.head_type == "lstm":
                sequence_output, _ = self.lstm(transaction_embeddings)

                representation = sequence_output[:, -1, :]

        logits = self.classifier(representation).squeeze(-1)
        return logits

In [65]:
def create_fraud_model(
    head_type,
    fine_tune_last_layers=0
):

    foundation_mlm = BertForMaskedLM.from_pretrained(
        "kunley2/FinBERT"
    )

    model = FinancialBERTFraudDetector(
        encoder=foundation_mlm.bert,
        hidden_size=foundation_mlm.config.hidden_size,
        head_type= head_type,
        lstm_hidden_size=128,
        dropout=0.2
    )

    if fine_tune_last_layers > 0:
        model.unfreeze_last_n_layers(fine_tune_last_layers)

    del foundation_mlm
    gc.collect()

    return model

In [66]:
train_labels_sampled = labels[train_targets]

In [67]:
positive_count = train_labels_sampled.sum()

negative_count = len(train_labels_sampled) - positive_count

In [68]:
POS_WEIGHT = negative_count / positive_count

print("Positive weight:", POS_WEIGHT)

Positive weight: 10.0


In [69]:
@torch.no_grad()
def predict(model, loader):
    model.eval()

    all_probabilities = []

    all_labels = []

    for batch in tqdm(loader, leave=False):
        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        logits = model(input_ids, attention_mask)

        probabilities = torch.sigmoid(logits).cpu().numpy()

        all_probabilities.append(probabilities)

        all_labels.append(batch["labels"].numpy())


    return (
        np.concatenate(all_labels),
        np.concatenate(all_probabilities)
    )

In [70]:
def find_best_threshold( y_true, probabilities):

    precision, recall, thresholds = (
        precision_recall_curve(y_true, probabilities)
    )


    precision = precision[:-1]

    recall = recall[:-1]


    f1 = (
        2 * precision * recall /
        np.maximum(precision + recall, 1e-12)
    )


    best = np.argmax(f1)

    return {
        "threshold": float(thresholds[best]),
        "precision": float(precision[best]),
        "recall": float(recall[best]),
        "f1": float(f1[best])
    }

In [71]:
def train_model(
    model,
    epochs=6,
    head_lr=3e-4,
    encoder_lr=1e-5,
    patience=2
):

    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            POS_WEIGHT,
            dtype=torch.float32,
            device=device
        )
    )


    head_parameters = []

    encoder_parameters = []


    for name, parameter in model.named_parameters():

        if not parameter.requires_grad:
            continue

        if name.startswith("encoder."):
            encoder_parameters.append(parameter)
        else:
            head_parameters.append(parameter)


    parameter_groups = [
        {
            "params": head_parameters,
            "lr": head_lr
        }
    ]

    if encoder_parameters:
        parameter_groups.append({
            "params": encoder_parameters,
            "lr": encoder_lr
        })


    optimizer = torch.optim.AdamW(
        parameter_groups,
        weight_decay=1e-4
    )


    best_ap = -1

    best_state = None

    no_improvement = 0


    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0
        examples = 0

        progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch}"
        )


        for batch in progress:

            input_ids = batch["input_ids"].to(device)


            attention_mask = batch["attention_mask"].to(device)

            y = batch["labels"].to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = model(input_ids, attention_mask)

            loss = criterion(logits, y)

            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                [
                    p for p in model.parameters() if p.requires_grad
                ],
                1.0
            )

            optimizer.step()

            size = input_ids.size(0)

            running_loss += loss.item() * size

            examples += size

            progress.set_postfix(
                loss=(
                    running_loss
                    / examples
                )
            )


        # Validation

        y_val, p_val = predict(model, validation_loader)


        val_ap = average_precision_score(y_val, p_val)

        val_auc = roc_auc_score(y_val, p_val)

        threshold_info = find_best_threshold(y_val, p_val)


        print(f"\nEpoch {epoch}")

        print(f"Validation AP: "f"{val_ap:.6f}")

        print(f"Validation ROC-AUC: "f"{val_auc:.6f}")

        print("Best validation:",threshold_info)

        if val_ap > best_ap:

            best_ap = val_ap


            best_state = {

                key:
                    value.detach()
                    .cpu()
                    .clone()

                for key, value
                in model
                .state_dict()
                .items()

            }
            no_improvement = 0

        else:
            no_improvement += 1

            if no_improvement >= patience:
                print("Early stopping.")
                break


    model.load_state_dict(best_state)

    return model

In [72]:
def evaluate_model(model):

    y_val, p_val = predict(model,validation_loader)

    threshold_info = find_best_threshold(y_val, p_val)

    threshold = threshold_info["threshold"]

    y_test, p_test = predict(model, test_loader)

    predictions = (
        p_test >= threshold
    ).astype(np.uint8)


    ap =  average_precision_score(y_test, p_test)


    roc_auc = roc_auc_score(y_test, p_test)

    precision = precision_score(y_test, predictions, zero_division=0)

    recall = recall_score(y_test, predictions, zero_division=0)

    f1 = f1_score(y_test, predictions, zero_division=0)


    cm = confusion_matrix(y_test, predictions)


    result = {
        "PR_AUC_AP": ap,
        "ROC_AUC": roc_auc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Threshold": threshold
    }


    print(result)


    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            predictions,
            digits=5,
            zero_division=0
        )
    )

    return result

## MLP

In [73]:
current_model = create_fraud_model(head_type="current_mlp")

config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 14.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

In [74]:
current_model = train_model(
    current_model
)

Epoch 1:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 1
Validation AP: 0.036787
Validation ROC-AUC: 0.951172
Best validation: {'threshold': 0.9725466966629028, 'precision': 0.06231454005934718, 'recall': 0.13953488372093023, 'f1': 0.08615384615384615}


Epoch 2:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]

...

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 3
Validation AP: 0.032361
Validation ROC-AUC: 0.958028
Best validation: {'threshold': 0.9813583493232727, 'precision': 0.05322338830584707, 'recall': 0.15725359911406422, 'f1': 0.07952954354522541}


Epoch 4:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 4
Validation AP: 0.034882
Validation ROC-AUC: 0.959187
Best validation: {'threshold': 0.9740960001945496, 'precision': 0.06523005241700641, 'recall': 0.12403100775193798, 'f1': 0.08549618320610687}
Early stopping.


In [75]:
current_results = evaluate_model(current_model)

  0%|          | 0/35118 [00:00<?, ?it/s]

  0%|          | 0/21097 [00:00<?, ?it/s]

{'PR_AUC_AP': np.float64(0.03981122755830851), 'ROC_AUC': np.float64(0.9637950911331395), 'Precision': 0.07105719237435008, 'Recall': 0.13078149920255183, 'F1': 0.09208309938236946, 'Threshold': 0.9845263957977295}

Confusion Matrix:
[[673394   1072]
 [   545     82]]

Classification Report:
              precision    recall  f1-score   support

         0.0    0.99919   0.99841   0.99880    674466
         1.0    0.07106   0.13078   0.09208       627

    accuracy                        0.99760    675093
   macro avg    0.53512   0.56460   0.54544    675093
weighted avg    0.99833   0.99760   0.99796    675093



In [76]:
del current_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Mean Pooling

In [77]:
mean_model = create_fraud_model(head_type="mean_mlp")

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

In [78]:
mean_model = train_model(mean_model)

Epoch 1:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 1
Validation AP: 0.004601
Validation ROC-AUC: 0.768502
Best validation: {'threshold': 0.914943277835846, 'precision': 0.013903743315508022, 'recall': 0.05758582502768549, 'f1': 0.022399310790437218}


Epoch 2:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]

...

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 5
Validation AP: 0.013236
Validation ROC-AUC: 0.789388
Best validation: {'threshold': 0.9746978282928467, 'precision': 0.02536824877250409, 'recall': 0.06866002214839424, 'f1': 0.03704810277860771}


Epoch 6:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 6
Validation AP: 0.013281
Validation ROC-AUC: 0.791259
Best validation: {'threshold': 0.9900041222572327, 'precision': 0.08118081180811808, 'recall': 0.024363233665559248, 'f1': 0.03747870528109029}


In [79]:
mean_results = (
    evaluate_model(mean_model)
)

  0%|          | 0/35118 [00:00<?, ?it/s]

  0%|          | 0/21097 [00:00<?, ?it/s]

{'PR_AUC_AP': np.float64(0.005888817230001305), 'ROC_AUC': np.float64(0.7705242007722941), 'Precision': 0.05, 'Recall': 0.011164274322169059, 'F1': 0.018252933507170794, 'Threshold': 0.9900041222572327}

Confusion Matrix:
[[674333    133]
 [   620      7]]

Classification Report:
              precision    recall  f1-score   support

         0.0    0.99908   0.99980   0.99944    674466
         1.0    0.05000   0.01116   0.01825       627

    accuracy                        0.99888    675093
   macro avg    0.52454   0.50548   0.50885    675093
weighted avg    0.99820   0.99888   0.99853    675093



In [80]:
del mean_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## LSTM

In [81]:
lstm_model = create_fraud_model(head_type="lstm")

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

In [82]:
lstm_model = train_model(
    lstm_model
)

Epoch 1:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 1
Validation AP: 0.078663
Validation ROC-AUC: 0.954200
Best validation: {'threshold': 0.9628581404685974, 'precision': 0.103125, 'recall': 0.18272425249169436, 'f1': 0.1318417898521774}


Epoch 2:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]

...

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 3
Validation AP: 0.056723
Validation ROC-AUC: 0.941608
Best validation: {'threshold': 0.9470387697219849, 'precision': 0.08164556962025317, 'recall': 0.14285714285714285, 'f1': 0.10390656463954893}


Epoch 4:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 4
Validation AP: 0.095751
Validation ROC-AUC: 0.960973
Best validation: {'threshold': 0.9978939890861511, 'precision': 0.14262295081967213, 'recall': 0.19269102990033224, 'f1': 0.16391898257183232}
Early stopping.


In [83]:
lstm_results = evaluate_model(lstm_model)

  0%|          | 0/35118 [00:00<?, ?it/s]

  0%|          | 0/21097 [00:00<?, ?it/s]

{'PR_AUC_AP': np.float64(0.06270707021034345), 'ROC_AUC': np.float64(0.9694063079478161), 'Precision': 0.10372129849564529, 'Recall': 0.20893141945773525, 'F1': 0.13862433862433862, 'Threshold': 0.9883987307548523}

Confusion Matrix:
[[673334   1132]
 [   496    131]]

Classification Report:
              precision    recall  f1-score   support

         0.0    0.99926   0.99832   0.99879    674466
         1.0    0.10372   0.20893   0.13862       627

    accuracy                        0.99759    675093
   macro avg    0.55149   0.60363   0.56871    675093
weighted avg    0.99843   0.99759   0.99799    675093



In [84]:
torch.save(
    lstm_model.state_dict(),
    "/content/frozen_bert_lstm_fraud.pt"
)

In [85]:
del lstm_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Tuning only the BERT final layer

In [86]:
finetune_model = (
    create_fraud_model(
        head_type="lstm",
        fine_tune_last_layers=1
    )
)

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

In [87]:
trainable = sum(

    p.numel()
    for p in
    finetune_model.parameters()
    if p.requires_grad
)


total = sum(

    p.numel()
    for p in
    finetune_model.parameters()
)

print(f"Trainable: {trainable:,}")

print(f"Total: {total:,}")

print(f"Percentage: "f"{100 * trainable / total:.2f}%")

Trainable: 338,689
Total: 3,812,609
Percentage: 8.88%


In [88]:
finetune_model = train_model(
    finetune_model,
    epochs=5,
    head_lr=1e-4,
    encoder_lr=1e-5
)

Epoch 1:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 1
Validation AP: 0.108700
Validation ROC-AUC: 0.964805
Best validation: {'threshold': 0.9920446872711182, 'precision': 0.1457326892109501, 'recall': 0.20044296788482835, 'f1': 0.16876456876456875}


Epoch 2:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]

...

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 4
Validation AP: 0.157755
Validation ROC-AUC: 0.980054
Best validation: {'threshold': 0.9942573308944702, 'precision': 0.21997621878715815, 'recall': 0.20487264673311184, 'f1': 0.2121559633027523}


Epoch 5:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 5
Validation AP: 0.167860
Validation ROC-AUC: 0.986389
Best validation: {'threshold': 0.9980602860450745, 'precision': 0.21364452423698385, 'recall': 0.26356589147286824, 'f1': 0.2359940505701537}


In [89]:
finetune_results = evaluate_model(finetune_model)

  0%|          | 0/35118 [00:00<?, ?it/s]

  0%|          | 0/21097 [00:00<?, ?it/s]

{'PR_AUC_AP': np.float64(0.1509534846369069), 'ROC_AUC': np.float64(0.9885361703668022), 'Precision': 0.21316165951359084, 'Recall': 0.23763955342902712, 'F1': 0.22473604826546004, 'Threshold': 0.9980602860450745}

Confusion Matrix:
[[673916    550]
 [   478    149]]

Classification Report:
              precision    recall  f1-score   support

         0.0    0.99929   0.99918   0.99924    674466
         1.0    0.21316   0.23764   0.22474       627

    accuracy                        0.99848    675093
   macro avg    0.60623   0.61841   0.61199    675093
weighted avg    0.99856   0.99848   0.99852    675093



In [98]:
comparison = pd.DataFrame([
    {

        "Model": "BERT Current Transaction + MLP",
        **current_results
    },
    {
        "Model": "BERT Whole-Sequence Mean + MLP",
        **mean_results
    },
    {
        "Model": "Frozen BERT + Transaction LSTM",
        **lstm_results
    },
    {

        "Model": "Last BERT Layer FT + LSTM",
        **finetune_results
    }
])

comparison

,Model,PR_AUC_AP,ROC_AUC,Precision,Recall,F1,Threshold
0,BERT Current Transaction + MLP,0.039811,0.963795,0.071057,0.130781,0.092083,0.984526
1,BERT Whole-Sequence Mean + MLP,0.005889,0.770524,0.050000,0.011164,0.018253,0.990004
2,Frozen BERT + Transaction LSTM,0.062707,0.969406,0.103721,0.208931,0.138624,0.988399
3,Last BERT Layer FT + LSTM,0.150953,0.988536,0.213162,0.237640,0.224736,0.998060


In [91]:
torch.cuda.empty_cache()

## Finetuning Two layers

In [92]:
finetune_model2 = (
    create_fraud_model(
        head_type="lstm",
        fine_tune_last_layers=2
    )
)

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

In [93]:
trainable = sum(
    p.numel()

    for p in
    finetune_model2.parameters()
    if p.requires_grad

)

total = sum(

    p.numel()

    for p in
    finetune_model2.parameters()
)

print(f"Trainable: {trainable:,}")

print(f"Total: {total:,}")

print(f"Percentage: "f"{100 * trainable / total:.2f}%")

Trainable: 536,961
Total: 3,812,609
Percentage: 14.08%


In [94]:
finetune_model2 = train_model(
    finetune_model2,
    epochs=5,
    head_lr=1e-4,
    encoder_lr=1e-5
)

Epoch 1:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 1
Validation AP: 0.168533
Validation ROC-AUC: 0.980302
Best validation: {'threshold': 0.9247586727142334, 'precision': 0.18961038961038962, 'recall': 0.3233665559246955, 'f1': 0.23905034793286942}


Epoch 2:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]

...

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 4
Validation AP: 0.230016
Validation ROC-AUC: 0.979569
Best validation: {'threshold': 0.9937241673469543, 'precision': 0.292806484295846, 'recall': 0.32004429678848284, 'f1': 0.30582010582010577}


Epoch 5:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 5
Validation AP: 0.182684
Validation ROC-AUC: 0.967941
Best validation: {'threshold': 0.9899682402610779, 'precision': 0.2946298984034833, 'recall': 0.2248062015503876, 'f1': 0.2550251256281407}
Early stopping.


In [95]:
finetune_results2 = evaluate_model(finetune_model2)

  0%|          | 0/35118 [00:00<?, ?it/s]

  0%|          | 0/21097 [00:00<?, ?it/s]

{'PR_AUC_AP': np.float64(0.23356101604480778), 'ROC_AUC': np.float64(0.9877651616891877), 'Precision': 0.32829046898638425, 'Recall': 0.34609250398724084, 'F1': 0.33695652173913043, 'Threshold': 0.9930599927902222}

Confusion Matrix:
[[674022    444]
 [   410    217]]

Classification Report:
              precision    recall  f1-score   support

         0.0    0.99939   0.99934   0.99937    674466
         1.0    0.32829   0.34609   0.33696       627

    accuracy                        0.99873    675093
   macro avg    0.66384   0.67272   0.66816    675093
weighted avg    0.99877   0.99873   0.99875    675093



# Finetunning for longer period

In [99]:
del finetune_model2

gc.collect()
torch.cuda.empty_cache()

In [100]:
finetune_model2 = (
    create_fraud_model(
        head_type="lstm",
        fine_tune_last_layers=2
    )
)

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

In [ ]:
finetune_model2 = train_model(
    finetune_model2,
    epochs=12,
    head_lr=1e-4,
    encoder_lr=1e-5,
    patience=5
)

Epoch 1:   0%|          | 0/2734 [00:00<?, ?it/s]

In [ ]:
finetune_results2 = evaluate_model(finetune_model2)